<div style="width: 100%; clear: both;">
  <div style="float: left; width: 50%;">
    <img src="https://www.ucc.edu.co/institucional/acerca-de-la-universidad/Documents/logo_ucc_2018(CURVAS)-01.png" align="left" style="max-width: 100%; height: auto;">
  </div>
  <div style="float: right; width: 50%;">
    <p style="margin: 0; padding-top: 22px; text-align:right;"><strong>Laboratorio de Tecnologias Emergentes</strong></p>
    <p style="margin: 0; text-align:right;"><strong>Linea de Inteligencia Artificial</strong></p>
    <p style="margin: 0; text-align:right;">Universidad Cooperativa de Colombia, Campus Ibague-Espinal</p>
    <p style="margin: 0; text-align:right;">Facultad de Ingenieria</p>
    <p style="margin: 0; text-align:right; padding-bottom: 100px;">Programa de Ingenieria de Sistemas</p>
  </div>
</div>
<div style="width:100%;">&nbsp;</div>
<div style="font-size: 20px; font-weight: bold; background: linear-gradient(to right, #ff7e5f, #feb47b); -webkit-background-clip: text; color: grey; text-align: center">W&F BirdLab -- 02 Preprocessing & Cropping</div>

# 03 - Dataset Split

Minimal notebook to validate `src/utils/split.py` and run the train/val/test split step.

Target split ratios: `75 / 15 / 10`


## 1. Setup
Load project paths, config, and the main split function.


In [1]:
import sys
from pathlib import Path
import yaml

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.split import split_dataset

CONFIG_PATH = PROJECT_ROOT / "configs" / "dataset.yaml"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Project root:", PROJECT_ROOT)
print("Config path:", CONFIG_PATH)
print("Configured split ratio:", config.get("split_ratio", [0.75, 0.15, 0.10]))


Project root: /home/walter/projects/aves
Config path: /home/walter/projects/aves/configs/dataset.yaml
Configured split ratio: [0.75, 0.15, 0.1]


## 2. Minimal synthetic test
Create a tiny fake accepted dataset, split it, and validate no leakage.


In [3]:
import shutil

tmp_root = PROJECT_ROOT / "data" / "_tmp_split_notebook"
tmp_input = tmp_root / "accepted"
tmp_output = tmp_root / "split"

if tmp_root.exists():
    shutil.rmtree(tmp_root)

(tmp_input / "class_a").mkdir(parents=True, exist_ok=True)
(tmp_input / "class_b").mkdir(parents=True, exist_ok=True)

for i in range(12):
    (tmp_input / "class_a" / f"a_{i}.jpg").write_bytes(b"x")
for i in range(5):
    (tmp_input / "class_b" / f"b_{i}.jpg").write_bytes(b"x")

summary_tmp = split_dataset(
    input_dir=tmp_input,
    output_dir=tmp_output,
    ratios=(0.75, 0.15, 0.10),
    seed=42,
)

print("Synthetic split summary:")
print(summary_tmp)

for cls in sorted(summary_tmp["per_class"]):
    split_sets = []
    for split_name in ("train", "val", "test"):
        names = {p.name for p in (tmp_output / split_name / cls).glob("*.jpg")}
        split_sets.append(names)
    leak = any(split_sets[i] & split_sets[j] for i in range(3) for j in range(i + 1, 3))
    print(f"{cls}: leakage={leak}, counts={summary_tmp['per_class'][cls]}")


Synthetic split summary:
{'input_dir': '/home/walter/projects/aves/data/_tmp_split_notebook/accepted', 'output_dir': '/home/walter/projects/aves/data/_tmp_split_notebook/split', 'ratios': {'train': 0.75, 'val': 0.15, 'test': 0.1}, 'seed': 42, 'use_symlinks': False, 'totals': {'train': 13, 'val': 2, 'test': 2}, 'per_class': {'class_a': {'train': 10, 'val': 1, 'test': 1}, 'class_b': {'train': 3, 'val': 1, 'test': 1}}, 'num_classes': 2, 'num_images': 17}
class_a: leakage=False, counts={'train': 10, 'val': 1, 'test': 1}
class_b: leakage=False, counts={'train': 3, 'val': 1, 'test': 1}


## 3. Example usage on pipeline data
Use accepted images from preprocessing and write split output to `data/split/`.


In [5]:
accepted_dir = PROJECT_ROOT / "data" / "processed"
split_dir = PROJECT_ROOT / "data" / "split"
ratios = tuple(config.get("split_ratio", [0.75, 0.15, 0.10]))

print("Accepted dir:", accepted_dir)
print("Split dir:", split_dir)
print("Ratios:", ratios)

if not accepted_dir.exists():
    print("Accepted directory does not exist yet. Run preprocessing first.")
else:
    summary = split_dataset(
        input_dir=accepted_dir,
        output_dir=split_dir,
        ratios=ratios,
        seed=42,
        use_symlinks=False,
    )
    print("\nFinal split summary:")
    print(summary)


Accepted dir: /home/walter/projects/aves/data/processed
Split dir: /home/walter/projects/aves/data/split
Ratios: (0.75, 0.15, 0.1)

Final split summary:
{'input_dir': '/home/walter/projects/aves/data/processed', 'output_dir': '/home/walter/projects/aves/data/split', 'ratios': {'train': 0.75, 'val': 0.15, 'test': 0.1}, 'seed': 42, 'use_symlinks': False, 'totals': {'train': 2771, 'val': 550, 'test': 361}, 'per_class': {'ardea-alba': {'train': 150, 'val': 30, 'test': 19}, 'atlapetes-latinuchus': {'train': 170, 'val': 34, 'test': 22}, 'butorides-striata': {'train': 167, 'val': 33, 'test': 22}, 'chlorochrysa-nitidissima': {'train': 205, 'val': 40, 'test': 27}, 'colibri-coruscans': {'train': 170, 'val': 34, 'test': 22}, 'common-gallinule': {'train': 153, 'val': 30, 'test': 20}, 'metallura-tyrianthina': {'train': 176, 'val': 35, 'test': 23}, 'momotus-aequatorialis': {'train': 275, 'val': 55, 'test': 36}, 'phimosus-infuscatus': {'train': 198, 'val': 39, 'test': 26}, 'piaya-cayana': {'train': 1

## 4. Optional cleanup
Remove temporary synthetic files created in section 2.


In [ ]:
# Uncomment if you want to remove synthetic test data
import shutil
if tmp_root.exists():
     shutil.rmtree(tmp_root)
     print("Removed:", tmp_root)
